# GPT-1 (openai-gpt) Dual Fine-tuning Experiment

이 노트북은 다음 실험을 수행합니다.

1. `GPT_A`: 감성 분류 데이터로 파인튜닝
2. `GPT_B`: QA 데이터로 파인튜닝
3. 두 모델 모두에 대해 감성/QA 입력을 넣고 결과를 비교

실행 환경: Google Colab + A100 GPU + High-RAM

In [1]:
# Colab 환경에서 최초 1회 실행
!pip -q install -U transformers datasets accelerate evaluate scikit-learn pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 150.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 151.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 135.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 48.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 whi

In [2]:
import os
import random
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

BASE_MODEL = "openai-gpt"  # GPT-1
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer vocab size:", tokenizer.vocab_size)
print("pad token:", tokenizer.pad_token)

device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/656 [00:00<?, ?B/s]

You are using a model of type `openai-gpt` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer vocab size: 40478
pad token: None


In [4]:
# -----------------------------
# 1) Sentiment dataset (SST-2)
# -----------------------------
# 런타임이 초기화되어도 이 셀만으로 tokenizer 상태를 복구
if "BASE_MODEL" not in globals():
    BASE_MODEL = "openai-gpt"
if "MAX_LEN" not in globals():
    MAX_LEN = 256
if "SEED" not in globals():
    SEED = 42

if "tokenizer" not in globals():
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

if tokenizer.pad_token is None:
    if tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token
    else:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})

print("Using pad token:", tokenizer.pad_token)
print("pad token id:", tokenizer.pad_token_id)

sst2 = load_dataset("glue", "sst2")

# Colab 실험 시간 단축을 위해 샘플링 (실제 split 크기를 넘지 않도록 자동 보정)
SENT_TRAIN_TARGET = 8000
SENT_VAL_TARGET = 1000

sent_train_size = len(sst2["train"])
sent_val_size = len(sst2["validation"])

SENT_TRAIN_N = min(SENT_TRAIN_TARGET, sent_train_size)
SENT_VAL_N = min(SENT_VAL_TARGET, sent_val_size)

print(f"SST-2 train size={sent_train_size}, using={SENT_TRAIN_N}")
print(f"SST-2 val size={sent_val_size}, using={SENT_VAL_N}")

sst2_train = sst2["train"].shuffle(seed=SEED).select(range(SENT_TRAIN_N))
sst2_val = sst2["validation"].shuffle(seed=SEED).select(range(SENT_VAL_N))

def preprocess_sst2(batch):
    return tokenizer(
        batch["sentence"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )

sst2_train_tok = sst2_train.map(preprocess_sst2, batched=True)
sst2_val_tok = sst2_val.map(preprocess_sst2, batched=True)

sst2_train_tok = sst2_train_tok.rename_column("label", "labels")
sst2_val_tok = sst2_val_tok.rename_column("label", "labels")

sst2_train_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
sst2_val_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(sst2_train_tok)
print(sst2_val_tok)

# -----------------------------
# 2) QA dataset (SQuAD v1)
# -----------------------------
squad = load_dataset("squad")

QA_TRAIN_TARGET = 20000
QA_VAL_TARGET = 2000

qa_train_size = len(squad["train"])
qa_val_size = len(squad["validation"])

QA_TRAIN_N = min(QA_TRAIN_TARGET, qa_train_size)
QA_VAL_N = min(QA_VAL_TARGET, qa_val_size)

print(f"SQuAD train size={qa_train_size}, using={QA_TRAIN_N}")
print(f"SQuAD val size={qa_val_size}, using={QA_VAL_N}")

qa_train = squad["train"].shuffle(seed=SEED).select(range(QA_TRAIN_N))
qa_val = squad["validation"].shuffle(seed=SEED).select(range(QA_VAL_N))


def preprocess_qa(batch):
    tokenized = tokenizer(
        batch["question"],
        batch["context"],
        truncation="only_second",
        max_length=MAX_LEN,
        padding="max_length",
        return_offsets_mapping=True,
    )

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(tokenized["offset_mapping"]):
        answer = batch["answers"][i]
        answer_start = answer["answer_start"][0]
        answer_text = answer["text"][0]
        answer_end = answer_start + len(answer_text)

        sequence_ids = tokenized.sequence_ids(i)

        context_start = 0
        while context_start < len(sequence_ids) and sequence_ids[context_start] != 1:
            context_start += 1

        context_end = len(sequence_ids) - 1
        while context_end >= 0 and sequence_ids[context_end] != 1:
            context_end -= 1

        if context_start >= len(sequence_ids) or context_end < 0:
            start_positions.append(0)
            end_positions.append(0)
            continue

        if offsets[context_start][0] > answer_start or offsets[context_end][1] < answer_end:
            start_positions.append(0)
            end_positions.append(0)
        else:
            start_idx = context_start
            while start_idx <= context_end and offsets[start_idx][0] <= answer_start:
                start_idx += 1
            start_positions.append(start_idx - 1)

            end_idx = context_end
            while end_idx >= context_start and offsets[end_idx][1] >= answer_end:
                end_idx -= 1
            end_positions.append(end_idx + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    tokenized.pop("offset_mapping")

    return tokenized

qa_train_tok = qa_train.map(preprocess_qa, batched=True, remove_columns=qa_train.column_names)
qa_val_tok = qa_val.map(preprocess_qa, batched=True, remove_columns=qa_val.column_names)

qa_train_tok.set_format(type="torch")
qa_val_tok.set_format(type="torch")

print(qa_train_tok)
print(qa_val_tok)

Using pad token: [PAD]
pad token id: 40478


sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

...

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

SQuAD train size=87599, using=20000
SQuAD val size=10570, using=2000


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 20000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 2000
})


In [6]:
# -----------------------------
# 3) Fine-tune GPT_A (Sentiment)
# -----------------------------
gpt_a = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)
gpt_a.resize_token_embeddings(len(tokenizer))
gpt_a.config.pad_token_id = tokenizer.pad_token_id

args_a = TrainingArguments(
    output_dir="./gpt_a_sentiment",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
    disable_tqdm=True,
)

trainer_a = Trainer(
    model=gpt_a,
    args=args_a,
    train_dataset=sst2_train_tok,
    eval_dataset=sst2_val_tok,
    data_collator=default_data_collator,
)

train_result_a = trainer_a.train()
print("GPT_A train metrics:", train_result_a.metrics)

trainer_a.save_model("./GPT_A")
tokenizer.save_pretrained("./GPT_A")
print("Saved GPT_A at ./GPT_A")

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

OpenAIGPTForSequenceClassification LOAD REPORT from: openai-gpt
Key                  | Status     | 
---------------------+------------+-
h.{0...11}.attn.bias | UNEXPECTED | 
score.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.5073', 'grad_norm': '5.274', 'learning_rate': '4.505e-05', 'epoch': '0.2'}
{'loss': '0.3726', 'grad_norm': '16.12', 'learning_rate': '4.005e-05', 'epoch': '0.4'}
{'loss': '0.3405', 'grad_norm': '13.5', 'learning_rate': '3.505e-05', 'epoch': '0.6'}
{'loss': '0.3287', 'grad_norm': '13.04', 'learning_rate': '3.005e-05', 'epoch': '0.8'}
{'loss': '0.294', 'grad_norm': '13.67', 'learning_rate': '2.505e-05', 'epoch': '1'}
{'eval_loss': '0.2772', 'eval_runtime': '1.198', 'eval_samples_per_second': '727.9', 'eval_steps_per_second': '45.91', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.1856', 'grad_norm': '12.79', 'learning_rate': '2.005e-05', 'epoch': '1.2'}
{'loss': '0.1698', 'grad_norm': '1.91', 'learning_rate': '1.505e-05', 'epoch': '1.4'}
{'loss': '0.1497', 'grad_norm': '1.778', 'learning_rate': '1.005e-05', 'epoch': '1.6'}
{'loss': '0.1459', 'grad_norm': '0.2147', 'learning_rate': '5.05e-06', 'epoch': '1.8'}
{'loss': '0.1545', 'grad_norm': '1.345', 'learning_rate': '5e-08', 'epoch': '2'}
{'eval_loss': '0.4616', 'eval_runtime': '1.203', 'eval_samples_per_second': '724.9', 'eval_steps_per_second': '45.72', 'epoch': '2'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '71.11', 'train_samples_per_second': '225', 'train_steps_per_second': '14.06', 'train_loss': '0.2649', 'epoch': '2'}
GPT_A train metrics: {'train_runtime': 71.1114, 'train_samples_per_second': 224.999, 'train_steps_per_second': 14.062, 'train_loss': 0.26485935592651366, 'epoch': 2.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved GPT_A at ./GPT_A


In [8]:
# -----------------------------
# 1) Sentiment dataset (SST-2)
# -----------------------------
# 런타임이 초기화되어도 이 셀만으로 tokenizer 상태를 복구
if "BASE_MODEL" not in globals():
    BASE_MODEL = "openai-gpt"
if "MAX_LEN" not in globals():
    MAX_LEN = 256
if "SEED" not in globals():
    SEED = 42

if "tokenizer" not in globals():
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

if tokenizer.pad_token is None:
    if tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token
    else:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})

print("Using pad token:", tokenizer.pad_token)
print("pad token id:", tokenizer.pad_token_id)

sst2 = load_dataset("glue", "sst2")

# Colab 실험 시간 단축을 위해 샘플링 (실제 split 크기를 넘지 않도록 자동 보정)
SENT_TRAIN_TARGET = 8000
SENT_VAL_TARGET = 1000

sent_train_size = len(sst2["train"])
sent_val_size = len(sst2["validation"])

SENT_TRAIN_N = min(SENT_TRAIN_TARGET, sent_train_size)
SENT_VAL_N = min(SENT_VAL_TARGET, sent_val_size)

print(f"SST-2 train size={sent_train_size}, using={SENT_TRAIN_N}")
print(f"SST-2 val size={sent_val_size}, using={SENT_VAL_N}")

sst2_train = sst2["train"].shuffle(seed=SEED).select(range(SENT_TRAIN_N))
sst2_val = sst2["validation"].shuffle(seed=SEED).select(range(SENT_VAL_N))

def preprocess_sst2(batch):
    return tokenizer(
        batch["sentence"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )

sst2_train_tok = sst2_train.map(preprocess_sst2, batched=True)
sst2_val_tok = sst2_val.map(preprocess_sst2, batched=True)

sst2_train_tok = sst2_train_tok.rename_column("label", "labels")
sst2_val_tok = sst2_val_tok.rename_column("label", "labels")

sst2_train_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
sst2_val_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(sst2_train_tok)
print(sst2_val_tok)

# -----------------------------
# 2) QA dataset (SQuAD v1) - Causal LM format
# -----------------------------
squad = load_dataset("squad")

QA_TRAIN_TARGET = 20000
QA_VAL_TARGET = 2000

qa_train_size = len(squad["train"])
qa_val_size = len(squad["validation"])

QA_TRAIN_N = min(QA_TRAIN_TARGET, qa_train_size)
QA_VAL_N = min(QA_VAL_TARGET, qa_val_size)

print(f"SQuAD train size={qa_train_size}, using={QA_TRAIN_N}")
print(f"SQuAD val size={qa_val_size}, using={QA_VAL_N}")

qa_train = squad["train"].shuffle(seed=SEED).select(range(QA_TRAIN_N))
qa_val = squad["validation"].shuffle(seed=SEED).select(range(QA_VAL_N))


def build_qa_text(example):
    answer_text = example["answers"]["text"][0] if len(example["answers"]["text"]) > 0 else ""
    prompt = (
        f"question: {example['question']}\n"
        f"context: {example['context']}\n"
        f"answer:"
    )
    full_text = f"{prompt} {answer_text}".strip()
    return {"prompt": prompt, "text": full_text}

qa_train_text = qa_train.map(build_qa_text)
qa_val_text = qa_val.map(build_qa_text)


def preprocess_qa_causal(batch):
    model_inputs = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )

    prompt_inputs = tokenizer(
        batch["prompt"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )

    labels = []
    for input_ids, prompt_ids in zip(model_inputs["input_ids"], prompt_inputs["input_ids"]):
        prompt_len = sum(1 for t in prompt_ids if t != tokenizer.pad_token_id)

        label_ids = input_ids.copy()
        for i in range(min(prompt_len, len(label_ids))):
            label_ids[i] = -100

        label_ids = [(-100 if t == tokenizer.pad_token_id else t) for t in label_ids]
        labels.append(label_ids)

    model_inputs["labels"] = labels
    return model_inputs

qa_train_tok = qa_train_text.map(
    preprocess_qa_causal,
    batched=True,
    remove_columns=qa_train_text.column_names,
)
qa_val_tok = qa_val_text.map(
    preprocess_qa_causal,
    batched=True,
    remove_columns=qa_val_text.column_names,
)

qa_train_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
qa_val_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(qa_train_tok)
print(qa_val_tok)

Using pad token: [PAD]
pad token id: 40478
SST-2 train size=67349, using=8000
SST-2 val size=872, using=872
Dataset({
    features: ['sentence', 'labels', 'idx', 'input_ids', 'attention_mask'],
    num_rows: 8000
})
Dataset({
    features: ['sentence', 'labels', 'idx', 'input_ids', 'attention_mask'],
    num_rows: 872
})
SQuAD train size=87599, using=20000
SQuAD val size=10570, using=2000


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 20000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2000
})


In [12]:
# -----------------------------
# 4) Fine-tune GPT_B (Question Answering as Causal LM)
# -----------------------------
if "AutoModelForCausalLM" not in globals():
    from transformers import AutoModelForCausalLM

if "BASE_MODEL" not in globals():
    BASE_MODEL = "openai-gpt"
if "tokenizer" not in globals():
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

gpt_b = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
gpt_b.resize_token_embeddings(len(tokenizer))
gpt_b.config.pad_token_id = tokenizer.pad_token_id

args_b = TrainingArguments(
    output_dir="./gpt_b_qa",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-5,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
    disable_tqdm=True,
)

trainer_b = Trainer(
    model=gpt_b,
    args=args_b,
    train_dataset=qa_train_tok,
    eval_dataset=qa_val_tok,
    data_collator=default_data_collator,
)

train_result_b = trainer_b.train()
print("GPT_B train metrics:", train_result_b.metrics)

trainer_b.save_model("./GPT_B")
tokenizer.save_pretrained("./GPT_B")
print("Saved GPT_B at ./GPT_B")

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

OpenAIGPTLMHeadModel LOAD REPORT from: openai-gpt
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


{'loss': '2.741', 'grad_norm': '24.57', 'learning_rate': '2.941e-05', 'epoch': '0.04'}
{'loss': '2.166', 'grad_norm': '29.93', 'learning_rate': '2.881e-05', 'epoch': '0.08'}
{'loss': '1.998', 'grad_norm': '27.97', 'learning_rate': '2.821e-05', 'epoch': '0.12'}
{'loss': '1.894', 'grad_norm': '38.84', 'learning_rate': '2.761e-05', 'epoch': '0.16'}
{'loss': '1.81', 'grad_norm': '39.24', 'learning_rate': '2.701e-05', 'epoch': '0.2'}
{'loss': '1.707', 'grad_norm': '33.82', 'learning_rate': '2.641e-05', 'epoch': '0.24'}
{'loss': '1.612', 'grad_norm': '21.38', 'learning_rate': '2.581e-05', 'epoch': '0.28'}
{'loss': '1.778', 'grad_norm': '32.52', 'learning_rate': '2.521e-05', 'epoch': '0.32'}
{'loss': '1.624', 'grad_norm': '44.72', 'learning_rate': '2.461e-05', 'epoch': '0.36'}
{'loss': '1.695', 'grad_norm': '30.98', 'learning_rate': '2.401e-05', 'epoch': '0.4'}
{'loss': '1.481', 'grad_norm': '32.56', 'learning_rate': '2.341e-05', 'epoch': '0.44'}
{'loss': '1.519', 'grad_norm': '66.29', 'learn

...

{'loss': '0.9497', 'grad_norm': '22.49', 'learning_rate': '1.441e-05', 'epoch': '1.04'}
{'loss': '0.9161', 'grad_norm': '30.79', 'learning_rate': '1.381e-05', 'epoch': '1.08'}
{'loss': '0.8818', 'grad_norm': '21.05', 'learning_rate': '1.321e-05', 'epoch': '1.12'}
{'loss': '0.8548', 'grad_norm': '15.8', 'learning_rate': '1.261e-05', 'epoch': '1.16'}
{'loss': '0.8398', 'grad_norm': '13.36', 'learning_rate': '1.201e-05', 'epoch': '1.2'}
{'loss': '0.8513', 'grad_norm': '32.79', 'learning_rate': '1.141e-05', 'epoch': '1.24'}
{'loss': '0.8957', 'grad_norm': '34.96', 'learning_rate': '1.081e-05', 'epoch': '1.28'}
{'loss': '0.9063', 'grad_norm': '26.8', 'learning_rate': '1.021e-05', 'epoch': '1.32'}
{'loss': '0.9227', 'grad_norm': '20.16', 'learning_rate': '9.606e-06', 'epoch': '1.36'}
{'loss': '0.8194', 'grad_norm': '21.45', 'learning_rate': '9.006e-06', 'epoch': '1.4'}
{'loss': '0.8292', 'grad_norm': '41.11', 'learning_rate': '8.406e-06', 'epoch': '1.44'}
{'loss': '0.802', 'grad_norm': '23.1

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '277.3', 'train_samples_per_second': '144.3', 'train_steps_per_second': '18.03', 'train_loss': '1.224', 'epoch': '2'}
GPT_B train metrics: {'train_runtime': 277.2571, 'train_samples_per_second': 144.27, 'train_steps_per_second': 18.034, 'train_loss': 1.2241306381225585, 'epoch': 2.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved GPT_B at ./GPT_B


In [13]:
import torch.nn.functional as F

if "AutoModelForCausalLM" not in globals():
    from transformers import AutoModelForCausalLM

id2label = {0: "negative", 1: "positive"}


def predict_sentiment(model, text):
    model.eval()
    model.to(device)
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LEN,
    ).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)[0].detach().cpu().numpy()
    pred = int(np.argmax(probs))
    return {
        "text": text,
        "pred_label": id2label.get(pred, str(pred)),
        "probs": probs.tolist(),
    }


def answer_question_generative(model, question, context, max_new_tokens=24):
    model.eval()
    model.to(device)
    prompt = f"question: {question}\ncontext: {context}\nanswer:"
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN,
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_ids = outputs[0][inputs["input_ids"].shape[1] :]
    answer = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    return {
        "question": question,
        "answer": answer,
    }


def predict_sentiment_generative(model, text, max_new_tokens=6):
    model.eval()
    model.to(device)
    prompt = f"review: {text}\nsentiment:"
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN,
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_ids = outputs[0][inputs["input_ids"].shape[1] :]
    raw = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    return {"text": text, "generated": raw}


# 실제 예시 입력
sentiment_examples = [
    "This movie is absolutely wonderful and touching.",
    "The product quality is terrible and I want a refund.",
]

qa_question = "Where do I live?"
qa_context = "My name is Minji, and I live in Seoul with my family."

print("=" * 80)
print("[GPT_A] sentiment fine-tuned model")
print("=" * 80)
for text in sentiment_examples:
    print("Sentiment input:", text)
    print("Sentiment output:", predict_sentiment(gpt_a, text))
    print()

print("QA input:")
print("Q:", qa_question)
print("C:", qa_context)
try:
    gpt_a_for_gen = AutoModelForCausalLM.from_pretrained("./GPT_A").to(device)
    qa_result_a = answer_question_generative(gpt_a_for_gen, qa_question, qa_context)
    print("QA output:", qa_result_a)
except Exception as e:
    print("QA output: GPT_A checkpoint cannot be loaded as CausalLM in this runtime.")
    print("Error:", str(e)[:300])

print("\n" + "=" * 80)
print("[GPT_B] QA fine-tuned model (Causal LM)")
print("=" * 80)
print("QA input:")
print("Q:", qa_question)
print("C:", qa_context)
print("QA output:", answer_question_generative(gpt_b, qa_question, qa_context))

print()
for text in sentiment_examples:
    print("Sentiment input:", text)
    print("Sentiment output (generative):", predict_sentiment_generative(gpt_b, text))
    print()

[GPT_A] sentiment fine-tuned model
Sentiment input: This movie is absolutely wonderful and touching.
Sentiment output: {'text': 'This movie is absolutely wonderful and touching.', 'pred_label': 'positive', 'probs': [0.000337979436153546, 0.9996620416641235]}

Sentiment input: The product quality is terrible and I want a refund.
Sentiment output: {'text': 'The product quality is terrible and I want a refund.', 'pred_label': 'negative', 'probs': [0.9990978240966797, 0.0009022055310197175]}

QA input:
Q: Where do I live?
C: My name is Minji, and I live in Seoul with my family.


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

OpenAIGPTLMHeadModel LOAD REPORT from: ./GPT_A
Key          | Status     |  | 
-------------+------------+--+-
score.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


QA output: {'question': 'Where do I live?', 'answer': "i do n't know . i do n't know . i do n't know . i do n't know . i do n't know"}

[GPT_B] QA fine-tuned model (Causal LM)
QA input:
Q: Where do I live?
C: My name is Minji, and I live in Seoul with my family.
QA output: {'question': 'Where do I live?', 'answer': 'seoul with my family . minji is the name of my family . minji is the name of my family .'}

Sentiment input: This movie is absolutely wonderful and touching.
Sentiment output (generative): {'text': 'This movie is absolutely wonderful and touching.', 'generated': 'sentiment : sentiment : sentiment :'}

Sentiment input: The product quality is terrible and I want a refund.
Sentiment output (generative): {'text': 'The product quality is terrible and I want a refund.', 'generated': 'i want a refund .'}



## 실행 가이드

1. 위에서부터 순서대로 실행
2. A100 + High-RAM 기준으로도 학습 시간이 걸릴 수 있음
3. 더 빠르게 보려면 `SENT_TRAIN_N`, `QA_TRAIN_N`, `num_train_epochs`를 줄이기
4. 더 정확하게 보려면 데이터 수와 epoch를 늘리기

참고:
- `GPT_A`는 감성 분류 head, `GPT_B`는 QA head라서 서로 다른 태스크를 직접 수행할 때 오류 또는 품질 저하가 날 수 있음
- 이 노트북은 "한 태스크로 파인튜닝한 모델이 다른 태스크에서 어떻게 보이는지"를 실험적으로 확인하기 위한 구성입니다.

## GPT_B_better 추가 학습

아래 셀은 QA 생성 품질 개선을 목표로 `GPT_B_better`를 학습합니다.

- 프롬프트/정답 포맷을 더 명확하게 구성
- 정답 토큰이 실제 라벨에 남는 샘플만 사용
- `GPT_B` 체크포인트가 있으면 이어서 파인튜닝, 없으면 `openai-gpt`부터 시작

In [14]:
# -----------------------------
# 5) Fine-tune GPT_B_better (Improved QA Causal LM)
# -----------------------------
if "AutoModelForCausalLM" not in globals():
    from transformers import AutoModelForCausalLM

if "BASE_MODEL" not in globals():
    BASE_MODEL = "openai-gpt"
if "SEED" not in globals():
    SEED = 42
if "MAX_LEN" not in globals():
    MAX_LEN = 256
if "tokenizer" not in globals():
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

if "qa_train" not in globals() or "qa_val" not in globals():
    squad = load_dataset("squad")
    QA_TRAIN_TARGET = 20000
    QA_VAL_TARGET = 2000
    qa_train_size = len(squad["train"])
    qa_val_size = len(squad["validation"])
    QA_TRAIN_N = min(QA_TRAIN_TARGET, qa_train_size)
    QA_VAL_N = min(QA_VAL_TARGET, qa_val_size)
    qa_train = squad["train"].shuffle(seed=SEED).select(range(QA_TRAIN_N))
    qa_val = squad["validation"].shuffle(seed=SEED).select(range(QA_VAL_N))


def build_qa_text_better(example):
    answer_text = example["answers"]["text"][0] if len(example["answers"]["text"]) > 0 else ""
    prompt = (
        f"question: {example['question']}\n"
        f"context: {example['context']}\n"
        f"short_answer:"
    )
    eos = tokenizer.eos_token if tokenizer.eos_token is not None else ""
    full_text = f"{prompt} {answer_text} {eos}".strip()
    return {"prompt": prompt, "text": full_text}


qa_train_text_better = qa_train.map(build_qa_text_better)
qa_val_text_better = qa_val.map(build_qa_text_better)


def preprocess_qa_causal_better(batch):
    model_inputs = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )

    prompt_inputs = tokenizer(
        batch["prompt"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )

    labels = []
    target_token_count = []

    for input_ids, prompt_ids in zip(model_inputs["input_ids"], prompt_inputs["input_ids"]):
        prompt_len = sum(1 for t in prompt_ids if t != tokenizer.pad_token_id)
        label_ids = input_ids.copy()

        for i in range(min(prompt_len, len(label_ids))):
            label_ids[i] = -100

        label_ids = [(-100 if t == tokenizer.pad_token_id else t) for t in label_ids]
        tgt_cnt = sum(1 for t in label_ids if t != -100)

        labels.append(label_ids)
        target_token_count.append(tgt_cnt)

    model_inputs["labels"] = labels
    model_inputs["target_token_count"] = target_token_count
    return model_inputs


qa_train_tok_better = qa_train_text_better.map(
    preprocess_qa_causal_better,
    batched=True,
    remove_columns=qa_train_text_better.column_names,
)
qa_val_tok_better = qa_val_text_better.map(
    preprocess_qa_causal_better,
    batched=True,
    remove_columns=qa_val_text_better.column_names,
)

# 정답 토큰이 너무 적은 샘플 제거 (잘린 샘플 완화)
qa_train_tok_better = qa_train_tok_better.filter(lambda x: x["target_token_count"] > 2)
qa_val_tok_better = qa_val_tok_better.filter(lambda x: x["target_token_count"] > 2)

qa_train_tok_better.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
qa_val_tok_better.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print("qa_train_tok_better:", qa_train_tok_better)
print("qa_val_tok_better:", qa_val_tok_better)

base_for_better = "./GPT_B" if os.path.isdir("./GPT_B") else BASE_MODEL
print("Start checkpoint for GPT_B_better:", base_for_better)

gpt_b_better = AutoModelForCausalLM.from_pretrained(base_for_better)
gpt_b_better.resize_token_embeddings(len(tokenizer))
gpt_b_better.config.pad_token_id = tokenizer.pad_token_id

args_b_better = TrainingArguments(
    output_dir="./gpt_b_better_qa",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
    disable_tqdm=True,
)

trainer_b_better = Trainer(
    model=gpt_b_better,
    args=args_b_better,
    train_dataset=qa_train_tok_better,
    eval_dataset=qa_val_tok_better,
    data_collator=default_data_collator,
)

train_result_b_better = trainer_b_better.train()
print("GPT_B_better train metrics:", train_result_b_better.metrics)

trainer_b_better.save_model("./GPT_B_better")
tokenizer.save_pretrained("./GPT_B_better")
print("Saved GPT_B_better at ./GPT_B_better")

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/20000 [00:00<?, ? examples/s]

...

{'loss': '0.347', 'grad_norm': '4.479', 'learning_rate': '6.017e-06', 'epoch': '2.099'}
{'loss': '0.3415', 'grad_norm': '3.675', 'learning_rate': '5.017e-06', 'epoch': '2.249'}
{'loss': '0.3102', 'grad_norm': '7.994', 'learning_rate': '4.018e-06', 'epoch': '2.399'}
{'loss': '0.3188', 'grad_norm': '4.795', 'learning_rate': '3.018e-06', 'epoch': '2.549'}
{'loss': '0.2973', 'grad_norm': '7.783', 'learning_rate': '2.019e-06', 'epoch': '2.699'}
{'loss': '0.3053', 'grad_norm': '7.134', 'learning_rate': '1.019e-06', 'epoch': '2.849'}
{'loss': '0.3119', 'grad_norm': '8.155', 'learning_rate': '1.999e-08', 'epoch': '2.999'}
{'eval_loss': '0.822', 'eval_runtime': '2.756', 'eval_samples_per_second': '390.8', 'eval_steps_per_second': '48.98', 'epoch': '3'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '218.3', 'train_samples_per_second': '146.6', 'train_steps_per_second': '9.165', 'train_loss': '0.4237', 'epoch': '3'}
GPT_B_better train metrics: {'train_runtime': 218.3359, 'train_samples_per_second': 146.609, 'train_steps_per_second': 9.165, 'train_loss': 0.42368488093425727, 'epoch': 3.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved GPT_B_better at ./GPT_B_better


In [15]:
# -----------------------------
# 6) Compare GPT_B vs GPT_B_better
# -----------------------------
if "AutoModelForCausalLM" not in globals():
    from transformers import AutoModelForCausalLM

if "BASE_MODEL" not in globals():
    BASE_MODEL = "openai-gpt"
if "MAX_LEN" not in globals():
    MAX_LEN = 256
if "device" not in globals():
    device = "cuda" if torch.cuda.is_available() else "cpu"
if "tokenizer" not in globals():
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token


def answer_question_generative_cmp(model, question, context, max_new_tokens=24):
    prompt = f"question: {question}\ncontext: {context}\nshort_answer:"
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN,
    ).to(device)

    model.eval()
    model.to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            no_repeat_ngram_size=3,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_ids = outputs[0][inputs["input_ids"].shape[1] :]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


def predict_sentiment_generative_cmp(model, text, max_new_tokens=6):
    prompt = f"review: {text}\nsentiment:"
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN,
    ).to(device)

    model.eval()
    model.to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            no_repeat_ngram_size=3,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_ids = outputs[0][inputs["input_ids"].shape[1] :]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


model_b = AutoModelForCausalLM.from_pretrained("./GPT_B").to(device)
model_b_better = AutoModelForCausalLM.from_pretrained("./GPT_B_better").to(device)

qa_cases = [
    {
        "q": "Where do I live?",
        "c": "My name is Minji, and I live in Seoul with my family.",
        "ref": "Seoul",
    },
    {
        "q": "Who wrote Hamlet?",
        "c": "Hamlet is a tragedy written by William Shakespeare sometime between 1599 and 1601.",
        "ref": "William Shakespeare",
    },
]

sentiment_cases = [
    "This movie is absolutely wonderful and touching.",
    "The product quality is terrible and I want a refund.",
]

print("=" * 90)
print("QA comparison: GPT_B vs GPT_B_better")
print("=" * 90)
for i, case in enumerate(qa_cases, start=1):
    out_b = answer_question_generative_cmp(model_b, case["q"], case["c"])
    out_b_better = answer_question_generative_cmp(model_b_better, case["q"], case["c"])
    print(f"[QA Case {i}]")
    print("Q:", case["q"])
    print("Ref:", case["ref"])
    print("GPT_B:", out_b)
    print("GPT_B_better:", out_b_better)
    print()

print("=" * 90)
print("Sentiment-style generation comparison: GPT_B vs GPT_B_better")
print("=" * 90)
for i, text in enumerate(sentiment_cases, start=1):
    out_b = predict_sentiment_generative_cmp(model_b, text)
    out_b_better = predict_sentiment_generative_cmp(model_b_better, text)
    print(f"[Sentiment Case {i}]")
    print("Input:", text)
    print("GPT_B:", out_b)
    print("GPT_B_better:", out_b_better)
    print()

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

QA comparison: GPT_B vs GPT_B_better
[QA Case 1]
Q: Where do I live?
Ref: Seoul
GPT_B: seoul is a small town in sekorea , and my family lives in sewla . seoul , the capital
GPT_B_better: seoul is a small town in sekorea , and seoul has been known to be a bit of a wild place

[QA Case 2]
Q: Who wrote Hamlet?
Ref: William Shakespeare
GPT_B: william shakespeare , jr. jr. jr. sr . jr. jr. . jr. . sr . sr - sr . 
 " william shakespeare sr
GPT_B_better: william shakespeare , jr. . william shakespeare is a tragic writer . william plays the piano concerto in the theater theater . william

Sentiment-style generation comparison: GPT_B vs GPT_B_better
[Sentiment Case 1]
Input: This movie is absolutely wonderful and touching.
GPT_B: sentiment : emotion : sentiment ,
GPT_B_better: sentiment : emotion : sentiment ,

[Sentiment Case 2]
Input: The product quality is terrible and I want a refund.
GPT_B: i do n't want a re
GPT_B_better: i do n't want a re

